In [1]:
import pandas as pd
from pathlib import Path
import csv
import time
import requests
from typing import List, Dict, Optional

In [2]:
#Function to slow down requests to avoid rate limit on the API
def fetch_with_retry(url, params, max_retries=5):
    for _ in range(max_retries):
        response = requests.get(url, params=params)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 20))
            print(f"Rate limited. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print(f"Error {response.status_code}: {response.text}")
            break
    return None


def fetch_historical_weather_multiple(
    latitudes: List[float],
    longitudes: List[float],
    start_date: str,
    end_date: str,
    target_variable: str,
    location_names: Optional[List[str]] = None,
) -> Dict[str, pd.DataFrame]:
    """
    Fetch historical weather data for multiple lat/lon pairs, including wind direction.

    Args:
        latitudes: List of latitudes.
        longitudes: List of longitudes.
        start_date: Start date in "YYYY-MM-DD" format.
        end_date: End date in "YYYY-MM-DD" format.
        location_names: Optional names for each location (default: "loc_0", "loc_1", ...).

    Returns:
        Dictionary of DataFrames (key: location name, value: weather data).
    """
    if len(latitudes) != len(longitudes):
        raise ValueError("Latitudes and longitudes must have the same length.")

    elif len(location_names) != len(latitudes):
        raise ValueError("Location names must match latitudes/longitudes length.")

    csv_path = Path('average_heating_days_cleaned.csv')
    file_exists = csv_path.exists()

    base_url = "https://archive-api.open-meteo.com/v1/archive"
    failed_locations = []
    counter = 1

    for lat, lon, name in zip(latitudes, longitudes, location_names):
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "daily": target_variable,
        }
        
        response = fetch_with_retry(base_url, params=params,max_retries=5)
        if response is None:         
            print(f"Failed to fetch data for {name} at ({lat}, {lon}). Skipping.")
            failed_locations.append((name, lat, lon))
            if len(failed_locations) >= 4:
                print("Too many failed locations. Stopping further requests.")
                return failed_locations
            continue
        print(f"{counter}. Successfully fetched data for {name} at ({lat}, {lon}).")
        df = pd.DataFrame(response['daily'], columns=['time', target_variable])

        
        heating_days_threshold = 18.0
        df['heating_day'] = df[target_variable] < heating_days_threshold
        average_for_location = df['heating_day'].mean() * 365

        # Write to CSV immediately after each location
        try:
            with open(csv_path, 'a', newline='') as f:
                writer = csv.writer(f)
                if not file_exists:
                    writer.writerow(['FSA', 'average_heating_days'])
                    file_exists = True
                try:
                    writer.writerow([name, average_for_location])
                except Exception as e:
                    print(f"Error writing to CSV for {name}: {e}")
        except Exception as e:
            print(f"Error opening CSV file for {name}: {e}")

        counter += 1
    return failed_locations

In [3]:
data_available_df = pd.read_csv('../radon/radon-concentration-cleaned.csv')
coordinates_df = pd.read_csv('../fsa_boundary/fsa_centroids.csv')
locations_df = coordinates_df[coordinates_df['FSA'].isin(data_available_df['FSA'])]


In [4]:
failed_locations_all = []
start_date = "2002-01-01"
end_date = "2011-12-31"
target_variable = "temperature_2m_max"

In [ ]:
# for i in range(0, len(locations_df), chunk_size):
#     locations = locations_df['FSA'].tolist()[i:i+chunk_size]
#     latitude = locations_df['latitude'].tolist()[i:i+chunk_size]
#     longitude = locations_df['longitude'].tolist()[i:i+chunk_size]

#     failed_locations = fetch_historical_weather_multiple(latitude, longitude,
#                     start_date, end_date,target_variable =target_variable,
#                     location_names=locations)
#     failed_locations_all.extend(failed_locations)
#     print(f"Chunk {i//chunk_size + 1} completed. Failed locations so far: {len(failed_locations_all)}."
#           f"Sleeping to avoid rate limits...")
#     time.sleep(300) 


# For testing with a my location
# locations = ['N6G']
# latitude = [42.9668]
# longitude = [-81.3049]

Manually fixing failed locations

In [ ]:
locations = locations_df['FSA'].tolist()[609:620]
latitude = locations_df['latitude'].tolist()[609:620]
longitude = locations_df['longitude'].tolist()[609:620]

failed_locations = fetch_historical_weather_multiple(latitude, longitude,
                start_date, end_date,target_variable =target_variable,
                location_names=locations)
failed_locations_all.extend(failed_locations)

Rate limited. Retrying in 20 seconds...


In [ ]:
chunk_size = 20
counter = 0
for i in range(620, len(locations_df), chunk_size):
    locations = locations_df['FSA'].tolist()[i:i+chunk_size]
    latitude = locations_df['latitude'].tolist()[i:i+chunk_size]
    longitude = locations_df['longitude'].tolist()[i:i+chunk_size]

    failed_locations = fetch_historical_weather_multiple(latitude, longitude,
                    start_date, end_date,target_variable =target_variable,
                    location_names=locations)
    failed_locations_all.extend(failed_locations)
    print(f"Chunk {i//chunk_size + 1} completed. Failed locations so far: {len(failed_locations_all)}."
          f"Sleeping to avoid rate limits...")
    counter += 1
    if counter % 3 == 0 and counter % 12 !=0: 
        time.sleep(300)  
    elif counter % 12 == 0:
        time.sleep(600)
    else:
        time.sleep(60) 

1. Successfully fetched data for L0P at (43.53351960809718, -79.96629896491403).
2. Successfully fetched data for L0R at (43.16081614638081, -79.7452208648283).
3. Successfully fetched data for L0S at (42.953928747282, -79.19333651591859).
4. Successfully fetched data for L1A at (43.979389837542776, -78.35939347129307).
5. Successfully fetched data for L1B at (43.92351233532928, -78.5457122614036).
6. Successfully fetched data for L1C at (43.97136446639861, -78.70952276461072).
7. Successfully fetched data for L1E at (43.8968884535182, -78.76833575011193).
8. Successfully fetched data for L1G at (43.92330385397188, -78.86843051474047).
9. Successfully fetched data for L1H at (43.972059742994965, -78.88365825106763).
10. Successfully fetched data for L1J at (43.86814178427262, -78.85406508942704).
11. Successfully fetched data for L1K at (43.93726262101392, -78.84454874277097).
12. Successfully fetched data for L1N at (43.87208825437643, -78.92731337868464).
13. Successfully fetched dat

KeyboardInterrupt: 